# Step 4: FedProto Baseline — Augmented Embeddings

FedProto (Tan et al., AAAI 2022) — prototype-based FL.
Key reviewer baseline. Shows PGA > FedProto under non-IID.

**Datasets:** `ham10000-vit-embeddings-aug`
**GPU T4 | ~90 min** (run parallel with Step 2)

Save output as: `feduca-phase4-aug-results`

In [ ]:
import os,json,time,copy,glob
import numpy as np
import torch,torch.nn as nn,torch.optim as optim
from torch.utils.data import DataLoader,TensorDataset
from scipy.stats import ttest_rel
import warnings; warnings.filterwarnings("ignore")

DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES=7; EMBED_DIM=768; ALPHAS=[1000,1,0.1]; SEEDS=[0,1,2,3,4]
CLASS_NAMES=['akiec','bcc','bkl','df','mel','nv','vasc']

hits=glob.glob("/kaggle/input/**/embeddings_train.npz",recursive=True)
aug=[h for h in hits if 'aug' in h]
EMBED_DIR=os.path.dirname(aug[0] if aug else hits[0])
print(f"Device:{DEVICE}  Embeddings:{EMBED_DIR}")

def load_split(name):
    d=np.load(f"{EMBED_DIR}/embeddings_{name}.npz",allow_pickle=True)
    return torch.tensor(d['embeddings'],dtype=torch.float32),torch.tensor(d['labels'],dtype=torch.long)

emb_train,lbl_train=load_split("train")
emb_val,  lbl_val  =load_split("val")
emb_test, lbl_test =load_split("test")
print(f"Train:{emb_train.shape} Val:{emb_val.shape} Test:{emb_test.shape}")

counts=torch.bincount(lbl_train,minlength=NUM_CLASSES).float()
weights=(1.0/counts); weights=(weights/weights.mean()).to(DEVICE)

class BottleneckMLP(nn.Module):
    def __init__(self,ratio=4):
        super().__init__(); b=768//ratio
        self.net=nn.Sequential(
            nn.Linear(768,4*b),nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,b),  nn.GELU(),nn.LayerNorm(b),  nn.Dropout(0.2),
            nn.Linear(b,4*b),  nn.GELU(),nn.LayerNorm(4*b),nn.Dropout(0.2),
            nn.Linear(4*b,7))
    def forward(self,x): return self.net(x)

@torch.no_grad()
def evaluate(head,embs,labels):
    head.eval(); p,t=[],[]
    for x,y in DataLoader(TensorDataset(embs,labels),batch_size=512):
        p.append(head(x.to(DEVICE)).argmax(1).cpu()); t.append(y)
    p=torch.cat(p).numpy(); t=torch.cat(t).numpy()
    f1s=[]
    for c in range(NUM_CLASSES):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum()
        pr=tp/(tp+fp+1e-9); rc=tp/(tp+fn+1e-9); f1s.append(float(2*pr*rc/(pr+rc+1e-9)))
    return {"accuracy":float((p==t).mean()),"macro_f1":float(np.mean(f1s)),
            "f1_per_class":[float(x) for x in f1s]}

def dirichlet_partition(labels,K,alpha,seed):
    rng=np.random.default_rng(seed); idx=[[] for _ in range(K)]
    for c in range(int(labels.max())+1):
        ci=np.where(labels.numpy()==c)[0]; rng.shuffle(ci)
        props=rng.dirichlet(np.full(K,alpha))
        splits=(np.cumsum(props)*len(ci)).astype(int)[:-1]
        for k,chunk in enumerate(np.split(ci,splits)): idx[k].extend(chunk.tolist())
    return [np.array(i) for i in idx]

def fedavg(states,sizes):
    total=sum(sizes)
    return {k:sum((sizes[i]/total)*states[i][k].float() for i in range(len(sizes))) for k in states[0]}

def local_train(head,embs,labels,lr=5e-3,epochs=5,bs=32):
    h=copy.deepcopy(head).to(DEVICE); h.train()
    opt=optim.Adam(h.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        for x,y in DataLoader(TensorDataset(embs.to(DEVICE),labels.to(DEVICE)),batch_size=bs,shuffle=True):
            opt.zero_grad(); crit(h(x),y).backward(); opt.step()
    return copy.deepcopy(h.state_dict())

def compute_prototypes(embs,labels):
    proto=torch.zeros(NUM_CLASSES,EMBED_DIM); count=torch.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        mask=labels==c
        if mask.sum()>0: proto[c]=embs[mask].mean(0); count[c]=float(mask.sum())
    return proto,count

def pga(states,sizes,client_protos,beta=5.0):
    K=len(states); total=sum(sizes)
    g_proto=torch.zeros(NUM_CLASSES,EMBED_DIM); g_count=torch.zeros(NUM_CLASSES)
    for k in range(K):
        pk,ck=client_protos[k]
        for c in range(NUM_CLASSES):
            if ck[c]>0: g_proto[c]+=ck[c]*pk[c]; g_count[c]+=ck[c]
    for c in range(NUM_CLASSES):
        if g_count[c]>0: g_proto[c]/=g_count[c]
    cos_d=lambda a,b: 1.0-float((a/(a.norm()+1e-9))@(b/(b.norm()+1e-9)))
    w=torch.zeros(NUM_CLASSES,K)
    for c in range(NUM_CLASSES):
        if g_proto[c].norm()<1e-9:
            for k in range(K): w[c,k]=sizes[k]/total
        else:
            lw=torch.zeros(K)
            for k in range(K):
                pk,ck=client_protos[k]
                lw[k]=-beta*cos_d(pk[c],g_proto[c]) if ck[c]>0 else -1e6
            w[c]=torch.softmax(lw,dim=0)
    fw=fb=None
    for key in states[0]:
        s=states[0][key].shape
        if len(s)==2 and s[0]==NUM_CLASSES: fw=key
        if len(s)==1 and s[0]==NUM_CLASSES: fb=key
    avg={}
    for key in states[0]:
        if key==fw:
            avg[key]=torch.stack([sum(w[c,k]*states[k][key][c].float() for k in range(K)) for c in range(NUM_CLASSES)])
        elif key==fb:
            avg[key]=torch.stack([sum(w[c,k]*states[k][key][c].float() for k in range(K)) for c in range(NUM_CLASSES)])
        else:
            avg[key]=sum((sizes[i]/total)*states[i][key].float() for i in range(K))
    return avg

print("All utilities ready.")

In [ ]:
RESULTS_DIR="/kaggle/working/results_phase4_aug"
os.makedirs(RESULTS_DIR,exist_ok=True)

def fedproto_local_train(head, embs, labels, global_protos,
                          lr=5e-3, epochs=5, bs=32, lam=0.1):
    """
    FedProto local training:
    Loss = CrossEntropy + lambda * ||local_proto - global_proto||^2
    Only classification head is trained; prototypes are class means
    of the penultimate layer (here we use embeddings directly).
    """
    h=copy.deepcopy(head).to(DEVICE); h.train()
    opt=optim.Adam(h.parameters(),lr=lr)
    crit=nn.CrossEntropyLoss(weight=weights)

    # Local prototypes from cached embeddings (fixed)
    local_protos={}
    for c in range(NUM_CLASSES):
        mask=(labels==c)
        if mask.sum()>0:
            local_protos[c]=embs[mask].mean(0).to(DEVICE)

    for _ in range(epochs):
        for x,y in DataLoader(TensorDataset(embs.to(DEVICE),labels.to(DEVICE)),
                               batch_size=bs,shuffle=True):
            opt.zero_grad()
            ce=crit(h(x),y)
            # Prototype regularisation
            proto_loss=torch.tensor(0.,device=DEVICE)
            for c,lp in local_protos.items():
                if c in global_protos:
                    proto_loss+=torch.norm(lp-global_protos[c].to(DEVICE))**2
            (ce + lam*proto_loss/NUM_CLASSES).backward()
            opt.step()
    return copy.deepcopy(h.state_dict())

def fedproto_aggregate(states, sizes, client_protos):
    """Aggregate: FedAvg weights + weighted average prototypes."""
    # Weight aggregation (standard FedAvg for head)
    agg_state = fedavg(states, sizes)
    # Global prototype = weighted mean of client prototypes
    total = sum(sizes)
    global_protos = {}
    for c in range(NUM_CLASSES):
        protos = [(client_protos[k][0][c], client_protos[k][1][c], sizes[k])
                  for k in range(len(states))
                  if client_protos[k][1][c] > 0]
        if protos:
            weighted = sum(p*n for p,n,sz in [(p,n,sz) for p,n,sz in protos])
            total_c  = sum(sz for _,_,sz in protos)
            global_protos[c] = weighted / total_c
    return agg_state, global_protos

print("FedProto utilities ready (Tan et al., AAAI 2022).")

fedproto_results=[]
print("="*55)
print("FedProto — Bottleneck N/4, 3 alpha, 5 seeds")
print("="*55)

for alpha in ALPHAS:
    for seed in SEEDS:
        partition=dirichlet_partition(lbl_train,K=4,alpha=alpha,seed=seed)
        c_embs  =[emb_train[idx] for idx in partition]
        c_labels=[lbl_train[idx] for idx in partition]
        sizes   =[len(idx) for idx in partition]

        head=BottleneckMLP(ratio=4).to(DEVICE)
        global_protos={}  # start empty
        best_val=-1; best_round=0; best_state=None
        t0=time.time()

        for rnd in range(1,61):
            states=[]; client_protos=[]
            for k in range(4):
                st=fedproto_local_train(head,c_embs[k],c_labels[k],global_protos)
                states.append(st)
                cp=compute_prototypes(c_embs[k],c_labels[k])
                client_protos.append(cp)

            agg_state, global_protos = fedproto_aggregate(states,sizes,client_protos)
            head.load_state_dict(agg_state)
            vm=evaluate(head,emb_val.to(DEVICE),lbl_val)
            if vm['macro_f1']>best_val:
                best_val=vm['macro_f1']; best_round=rnd
                best_state=copy.deepcopy(head.state_dict())

        head.load_state_dict(best_state)
        tm=evaluate(head,emb_test.to(DEVICE),lbl_test)
        res={"method":"fedproto","alpha":alpha,"seed":seed,
             "best_round":best_round,"test":tm}
        fedproto_results.append(res)
        with open(os.path.join(RESULTS_DIR,f"fedproto_a{alpha}_s{seed}.json"),"w") as f:
            json.dump(res,f,indent=2)

    f1s=[r['test']['macro_f1'] for r in fedproto_results if r['alpha']==alpha]
    print(f"  FedProto alpha={alpha}: F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}")

with open(os.path.join(RESULTS_DIR,"fedproto_all.json"),"w") as f:
    json.dump(fedproto_results,f,indent=2)
print("\nFedProto results saved.")
print("Save output as: feduca-phase4-aug-results")